# NEXUS-RAG: Retrieval Analysis

Analyse dense, sparse, and hybrid retrieval performance.

In [ ]:
import sys; sys.path.insert(0, "..")
from retrieval.hybrid_retriever import HybridRetriever
retriever = HybridRetriever()
query = "What is Retrieval-Augmented Generation?"
results = retriever.retrieve(query, k=10)
for r in results:
    print(f"{r.score:.3f}  [{r.source}]  {r.content[:80]}")

In [ ]:
# Score distribution: dense vs BM25
from retrieval.dense_retriever import DenseRetriever
from retrieval.sparse_retriever import SparseRetriever
import plotly.graph_objects as go
dense = DenseRetriever().search(query, k=20)
bm25 = SparseRetriever().search(query, k=20)
fig = go.Figure()
fig.add_trace(go.Histogram(x=[r.score for r in dense], name="Dense", opacity=0.7))
fig.add_trace(go.Histogram(x=[r.score for r in bm25], name="BM25", opacity=0.7))
fig.update_layout(barmode="overlay", title="Dense vs BM25 Score Distribution")
fig.show()

In [ ]:
# Reranking score shift
from ingestion.indexer import ScoredDocument
from retrieval.reranker import CrossEncoderReranker
ce = CrossEncoderReranker()
reranked = ce.rerank(query, dense, top_k=10)
fig = go.Figure()
fig.add_trace(go.Bar(name="Pre-CE", x=list(range(10)), y=[r.score for r in dense[:10]]))
fig.add_trace(go.Bar(name="Post-CE", x=list(range(10)), y=[r.score for r in reranked]))
fig.update_layout(barmode="group", title="CrossEncoder Reranking Score Shift")
fig.show()